Given a 2D grid grid where '1' represents land and '0' represents water, count and return the number of islands.

An island is formed by connecting adjacent lands horizontally or vertically and is surrounded by water. You may assume water is surrounding the grid (i.e., all the edges are water).

Example 1:

Input: grid = [

    ["0","1","1","1","0"],

    ["0","1","0","1","0"],

    ["1","1","0","0","0"],

    ["0","0","0","0","0"]

  ]

Output: 1

Example 2:

Input: grid = [

    ["1","1","0","0","1"],

    ["1","1","0","0","1"],

    ["0","0","1","0","0"],

    ["0","0","0","1","1"]
    
  ]

Output: 4

In [ ]:
from typing import Optional, List
from collections import deque

# DFS

Intuition

Think of the grid as a map where '1' is land and '0' is water.

An island is a group of connected land cells (up, down, left, right).

Whenever we find a land cell that hasn’t been visited, we start a DFS to sink the entire island by marking all its connected land as water. Each DFS call corresponds to one island.

Algorithm

Iterate through every cell in the grid.

When a cell with value '1' is found:

Increment the island count.

Run DFS from that cell.

In DFS:

If the cell is out of bounds or is '0', return.

Mark the current cell as '0' (visited).

Recursively explore all 4 directions (up, down, left, right).

Continue until all cells are processed.

Return the total island count.

In [ ]:
# Time Complexity: O(m × n)
# Space Complexity: O(m × n)
class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
        directions = [[1, 0], [-1, 0], [0, 1], [0, -1]]
        ROWS, COLS = len(grid), len(grid[0])
        islands = 0

        def dfs(r, c):
            if (r < 0 or c < 0 or r >= ROWS or
                c >= COLS or grid[r][c] == "0"
            ):
                return

            grid[r][c] = "0"
            for dr, dc in directions:
                dfs(r + dr, c + dc)

        for r in range(ROWS):
            for c in range(COLS):
                if grid[r][c] == "1":
                    dfs(r, c)
                    islands += 1

        return islands

# BFS

Intuition

Treat the grid like a map where '1' represents land and '0' represents water.

Each island is a group of connected land cells.

When we encounter a land cell, we use BFS to visit all connected land cells and mark them as water, ensuring the same island is not counted again.

Algorithm

Traverse every cell in the grid.

When a '1' (land) cell is found:

Increment the island count.

Start BFS from that cell.

In BFS:

Push the starting cell into a queue and mark it as '0'.

While the queue is not empty:

Pop a cell.

Explore its 4 neighbors (up, down, left, right).

If a neighbor is land, mark it as '0' and add it to the queue.

Continue scanning the grid.

Return the total number of islands.

In [ ]:
# Time Complexity: O(m × n)
# Space Complexity: O(m × n)
class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
        directions = [[1, 0], [-1, 0], [0, 1], [0, -1]]
        ROWS, COLS = len(grid), len(grid[0])
        islands = 0

        def bfs(r, c):
            q = deque()
            grid[r][c] = "0"
            q.append((r, c))

            while q:
                row, col = q.popleft()
                for dr, dc in directions:
                    nr, nc = dr + row, dc + col
                    if (nr < 0 or nc < 0 or nr >= ROWS or
                        nc >= COLS or grid[nr][nc] == "0"
                    ):
                        continue
                    q.append((nr, nc))
                    grid[nr][nc] = "0"

        for r in range(ROWS):
            for c in range(COLS):
                if grid[r][c] == "1":
                    bfs(r, c)
                    islands += 1

        return islands

# Disjoint Set Union

Intuition

Think of every land cell ('1') as its own separate island initially.

When two land cells are adjacent (up, down, left, right), they actually belong to the same island, so we should merge them.

Disjoint Set Union (Union-Find) helps us:

Quickly connect adjacent land cells

Avoid counting the same island multiple times

Each successful merge reduces the total island count by 1.

Algorithm

Treat each cell as a node and map (row, col) to a unique index.

Initialize DSU for all cells.

Traverse the grid:

If a cell is land ('1'), increment island count.

Check its 4 neighbors.

If a neighbor is also land:

Union the two cells.

If a union actually happens, decrement island count.

After processing all cells, the remaining count is the number of islands.

Return the island count.


In [ ]:
# Time Complexity: O(m × n)
# Space Complexity: O(m × n)
class DSU:
    def __init__(self, n):
        self.Parent = list(range(n + 1))
        self.Size = [1] * (n + 1)

    def find(self, node):
        if self.Parent[node] != node:
            self.Parent[node] = self.find(self.Parent[node])
        return self.Parent[node]

    def union(self, u, v):
        pu = self.find(u)
        pv = self.find(v)
        if pu == pv:
            return False
        if self.Size[pu] >= self.Size[pv]:
            self.Size[pu] += self.Size[pv]
            self.Parent[pv] = pu
        else:
            self.Size[pv] += self.Size[pu]
            self.Parent[pu] = pv
        return True

class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
        ROWS, COLS = len(grid), len(grid[0])
        dsu = DSU(ROWS * COLS)

        def index(r, c):
            return r * COLS + c

        directions = [(1, 0), (-1, 0), (0, 1), (0, -1)]
        islands = 0

        for r in range(ROWS):
            for c in range(COLS):
                if grid[r][c] == '1':
                    islands += 1
                    for dr, dc in directions:
                        nr, nc = r + dr, c + dc
                        if (nr < 0 or nc < 0 or nr >= ROWS or
                            nc >= COLS or grid[nr][nc] == "0"
                        ):
                            continue

                        if dsu.union(index(r, c), index(nr, nc)):
                            islands -= 1

        return islands